# **BLOCK 1: SETUP, UNZIP DATASET, AND VERIFICATION**

In [10]:
# ============================================================================
# UPDATED BLOCK 1: SETUP, UNZIP, AND VERIFY STRUCTURE
# ============================================================================

import os
import json
import yaml
import zipfile
from pathlib import Path

# Dataset paths
BASE_DATASET_PATH = "/kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection"
WORKING_DIR = "/kaggle/working"

print("="*70)
print("HIPS & PELVIS FRACTURE DETECTION - FASTER R-CNN")
print("="*70)
print(f"📁 Base dataset path: {BASE_DATASET_PATH}")

# ========== STEP 1: CHECK AVAILABLE FOLDERS ==========
print("\n🔍 Checking available folders...")
!ls -la "{BASE_DATASET_PATH}"

# ========== STEP 2: LOCATE HIPS AND PELVIS DATASET FOLDER ==========
# The main dataset folder with train/valid/test
dataset_folder = os.path.join(BASE_DATASET_PATH, "Hips_and_Pelvis", "Hips and Pelvis")

if not os.path.exists(dataset_folder):
    # Try alternative paths
    alt_paths = [
        os.path.join(BASE_DATASET_PATH, "Hips_and_Pelvis"),
        os.path.join(BASE_DATASET_PATH, "Hips and Pelvis"),
        os.path.join(BASE_DATASET_PATH, "Hips_and_Pelvis", "Hips and Pelvis"),
    ]
    for alt in alt_paths:
        if os.path.exists(alt):
            dataset_folder = alt
            break

print(f"\n📁 Dataset folder: {dataset_folder}")

if os.path.exists(dataset_folder):
    print("✅ Dataset folder found")
    print("\n📂 Dataset structure:")
    !ls -la "{dataset_folder}"
else:
    print("❌ Dataset folder not found!")

# ========== STEP 3: LOCATE YAML FILES FOLDER ==========
yaml_folder = os.path.join(BASE_DATASET_PATH, "Hips and Pelvis ymal Files")

if not os.path.exists(yaml_folder):
    # Try alternative names
    alt_yaml = [
        os.path.join(BASE_DATASET_PATH, "Hips and Pelvis yaml Files"),
        os.path.join(BASE_DATASET_PATH, "Hips_and_Pelvis_ymal_Files"),
    ]
    for alt in alt_yaml:
        if os.path.exists(alt):
            yaml_folder = alt
            break

print(f"\n📁 YAML folder: {yaml_folder}")

if os.path.exists(yaml_folder):
    print("✅ YAML folder found")
    print("\n📂 YAML structure:")
    !ls -la "{yaml_folder}"
    
    # List subfolders
    for subdir in os.listdir(yaml_folder):
        subdir_path = os.path.join(yaml_folder, subdir)
        if os.path.isdir(subdir_path):
            print(f"\n  📁 {subdir}:")
            !ls -la "{subdir_path}"
else:
    print("❌ YAML folder not found!")

# ========== STEP 4: FIND TRAIN/VAL/TEST PATHS ==========
print("\n🔍 Finding train/valid/test splits...")

train_path = None
val_path = None
test_path = None

# Look for train, valid, test folders
for folder_name in ['train', 'valid', 'val', 'test']:
    folder_path = os.path.join(dataset_folder, folder_name)
    if os.path.exists(folder_path):
        if 'train' in folder_name:
            train_path = folder_path
            print(f"  ✅ Train found: {folder_path}")
        elif 'valid' in folder_name or 'val' in folder_name:
            val_path = folder_path
            print(f"  ✅ Validation found: {folder_path}")
        elif 'test' in folder_name:
            test_path = folder_path
            print(f"  ✅ Test found: {folder_path}")

# If not found, check inside subfolders
if not train_path:
    for subdir in os.listdir(dataset_folder):
        subdir_path = os.path.join(dataset_folder, subdir)
        if os.path.isdir(subdir_path):
            if 'train' in subdir.lower():
                train_path = subdir_path
            elif 'valid' in subdir.lower() or 'val' in subdir.lower():
                val_path = subdir_path
            elif 'test' in subdir.lower():
                test_path = subdir_path

print(f"\n📊 Split paths:")
print(f"  Train: {train_path if train_path else '❌ Not found'}")
print(f"  Validation: {val_path if val_path else '❌ Not found'}")
print(f"  Test: {test_path if test_path else '❌ Not found'}")

# ========== STEP 5: GET IMAGES AND LABELS PATHS ==========
def get_images_labels(split_path, split_name):
    """Get images and labels paths for a split"""
    if not split_path:
        return None, None
    
    images_path = os.path.join(split_path, 'images')
    labels_path = os.path.join(split_path, 'labels')
    
    # If not found, check if they're directly in the split folder
    if not os.path.exists(images_path):
        images_path = split_path
    if not os.path.exists(labels_path):
        # Look for labels folder
        for item in os.listdir(split_path):
            item_path = os.path.join(split_path, item)
            if os.path.isdir(item_path) and 'label' in item.lower():
                labels_path = item_path
                break
    
    # Count files
    if os.path.exists(images_path):
        images = len([f for f in os.listdir(images_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {split_name} images: {images}")
    else:
        print(f"  ❌ {split_name} images not found")
        images_path = None
    
    if os.path.exists(labels_path):
        labels = len([f for f in os.listdir(labels_path) if f.endswith('.txt')])
        print(f"  {split_name} labels: {labels}")
    else:
        print(f"  ❌ {split_name} labels not found")
        labels_path = None
    
    return images_path, labels_path

train_images, train_labels = get_images_labels(train_path, "TRAIN")
val_images, val_labels = get_images_labels(val_path, "VALIDATION")
test_images, test_labels = get_images_labels(test_path, "TEST")

# ========== STEP 6: SAVE PATHS ==========
with open(f"{WORKING_DIR}/paths.txt", 'w') as f:
    f.write(f"DATASET_FOLDER={dataset_folder}\n")
    f.write(f"YAML_FOLDER={yaml_folder}\n")
    f.write(f"TRAIN_IMAGES={train_images}\n")
    f.write(f"TRAIN_LABELS={train_labels}\n")
    f.write(f"VAL_IMAGES={val_images}\n")
    f.write(f"VAL_LABELS={val_labels}\n")
    f.write(f"TEST_IMAGES={test_images}\n")
    f.write(f"TEST_LABELS={test_labels}\n")
    f.write(f"WORKING_DIR={WORKING_DIR}\n")

print(f"\n✅ Paths saved to {WORKING_DIR}/paths.txt")

print("\n" + "="*70)
print("✅ UPDATED BLOCK 1 COMPLETE")
print("="*70)

HIPS & PELVIS FRACTURE DETECTION - FASTER R-CNN
📁 Base dataset path: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection

🔍 Checking available folders...
total 4
drwxr-xr-x 4 nobody nogroup    0 Mar 23 23:13  .
drwxr-xr-x 3 root   root    4096 Mar 23 23:16  ..
drwxr-xr-x 3 nobody nogroup    0 Mar 23 23:13  Hips_and_Pelvis
drwxr-xr-x 4 nobody nogroup    0 Mar 23 23:13 'Hips and Pelvis ymal Files'

📁 Dataset folder: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis
✅ Dataset folder found

📂 Dataset structure:
total 0
drwxr-xr-x 5 nobody nogroup 0 Mar 23 23:16 .
drwxr-xr-x 3 nobody nogroup 0 Mar 23 23:13 ..
drwxr-xr-x 4 nobody nogroup 0 Mar 23 23:13 test
drwxr-xr-x 4 nobody nogroup 0 Mar 23 23:15 train
drwxr-xr-x 4 nobody nogroup 0 Mar 23 23:16 valid

📁 YAML folder: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips and Pelvis ymal Files
✅ YAML folder found

📂 YAML structure:
to

# **BLOCK 2: INSTALL FASTER R-CNN DEPENDENCIES**

In [11]:
# ============================================================================
# UPDATED BLOCK 2: INSTALL FASTER R-CNN DEPENDENCIES
# ============================================================================

print("="*70)
print("INSTALLING FASTER R-CNN DEPENDENCIES")
print("="*70)

# Install required packages
!pip install -q pandas pyyaml scikit-learn tqdm matplotlib opencv-python

# Install Detectron2 (Faster R-CNN implementation)
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2
from PIL import Image

print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n" + "="*70)
print("✅ UPDATED BLOCK 2 COMPLETE")
print("="*70)

INSTALLING FASTER R-CNN DEPENDENCIES
  Preparing metadata (setup.py) ... done

✅ PyTorch: 2.10.0+cu128
✅ CUDA Available: True
✅ GPU: Tesla T4
✅ GPU Memory: 15.6 GB

✅ UPDATED BLOCK 2 COMPLETE


# **BLOCK 3: ANALYZE DATASET CLASSES**

In [12]:
# ============================================================================
# UPDATED BLOCK 3: LOAD CLASS NAMES FROM YAML FILES
# ============================================================================

print("="*70)
print("LOADING CLASS NAMES FROM YAML FILES")
print("="*70)

# Load paths
with open(f"{WORKING_DIR}/paths.txt", 'r') as f:
    for line in f:
        if line.startswith("YAML_FOLDER"):
            YAML_FOLDER = line.strip().split('=')[1]
        if line.startswith("TRAIN_LABELS"):
            TRAIN_LABELS = line.strip().split('=')[1]
        if line.startswith("VAL_LABELS"):
            VAL_LABELS = line.strip().split('=')[1]
        if line.startswith("TEST_LABELS"):
            TEST_LABELS = line.strip().split('=')[1]

print(f"📁 YAML folder: {YAML_FOLDER}")

# Collect all class names from YAML files
all_classes = []
dataset_class_maps = {}

def load_yaml_classes(yaml_path, dataset_name):
    """Load class names from a YAML file"""
    try:
        with open(yaml_path, 'r') as f:
            data = yaml.safe_load(f)
            names = data.get('names', [])
            
            if isinstance(names, dict):
                names = [names[i] for i in sorted(names.keys())]
            
            print(f"\n📁 {dataset_name}:")
            for i, name in enumerate(names):
                print(f"  Class {i}: {name}")
            
            return names
    except Exception as e:
        print(f"⚠️ Error loading {yaml_path}: {e}")
        return []

# Search for YAML files in the YAML folder
if os.path.exists(YAML_FOLDER):
    for item in os.listdir(YAML_FOLDER):
        item_path = os.path.join(YAML_FOLDER, item)
        if os.path.isdir(item_path):
            # Look for data.yaml inside
            yaml_file = os.path.join(item_path, 'data.yaml')
            if os.path.exists(yaml_file):
                names = load_yaml_classes(yaml_file, item)
                dataset_class_maps[item] = names
                all_classes.extend(names)
else:
    print(f"❌ YAML folder not found: {YAML_FOLDER}")

# Get unique classes
unique_classes = sorted(list(set(all_classes)))

print(f"\n📊 TOTAL UNIQUE CLASSES: {len(unique_classes)}")
print("\n📋 Master class list:")
for i, class_name in enumerate(unique_classes):
    print(f"  {i}: {class_name}")

# Save class names
with open(f"{WORKING_DIR}/class_names.json", 'w') as f:
    json.dump(unique_classes, f, indent=2)

# Create class list
class_list = unique_classes

print(f"\n✅ Class names saved to {WORKING_DIR}/class_names.json")

print("\n" + "="*70)
print("✅ UPDATED BLOCK 3 COMPLETE")
print("="*70)

LOADING CLASS NAMES FROM YAML FILES
📁 YAML folder: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips and Pelvis ymal Files

📁 Pelvis Fracture .v3i.yolov8:
  Class 0: pelvic_fracture

📁 hip.v12-rotation_shear.yolov8:
  Class 0: fracture
  Class 1: no fracture

📊 TOTAL UNIQUE CLASSES: 3

📋 Master class list:
  0: fracture
  1: no fracture
  2: pelvic_fracture

✅ Class names saved to /kaggle/working/class_names.json

✅ UPDATED BLOCK 3 COMPLETE


# **BLOCK 4: CONVERT YOLO TO COCO FORMAT (FOR FASTER R-CNN)**

In [13]:
# ============================================================================
# UPDATED BLOCK 4: CONVERT YOLO TO COCO FORMAT (WITH CLASS MAPPING)
# ============================================================================

print("="*70)
print("CONVERTING YOLO TO COCO FORMAT")
print("="*70)

import json
from PIL import Image

# Load paths
with open(f"{WORKING_DIR}/paths.txt", 'r') as f:
    for line in f:
        if line.startswith("TRAIN_IMAGES"):
            TRAIN_IMAGES = line.strip().split('=')[1]
        if line.startswith("TRAIN_LABELS"):
            TRAIN_LABELS = line.strip().split('=')[1]
        if line.startswith("VAL_IMAGES"):
            VAL_IMAGES = line.strip().split('=')[1]
        if line.startswith("VAL_LABELS"):
            VAL_LABELS = line.strip().split('=')[1]
        if line.startswith("TEST_IMAGES"):
            TEST_IMAGES = line.strip().split('=')[1]
        if line.startswith("TEST_LABELS"):
            TEST_LABELS = line.strip().split('=')[1]

# Load class list
with open(f"{WORKING_DIR}/class_names.json", 'r') as f:
    class_list = json.load(f)

print(f"📋 Classes for Faster R-CNN ({len(class_list)} classes):")
for i, name in enumerate(class_list):
    print(f"  {i}: {name}")

# Create class name to ID mapping
class_to_id = {name: i for i, name in enumerate(class_list)}

def yolo_to_coco(images_dir, labels_dir, output_json, class_list, class_to_id, split_name):
    """
    Convert YOLO format to COCO JSON format for a specific split
    """
    if not images_dir or not os.path.exists(images_dir):
        print(f"⚠️ {split_name}: Images directory not found")
        return None
    
    if not labels_dir or not os.path.exists(labels_dir):
        print(f"⚠️ {split_name}: Labels directory not found")
        return None
    
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [{"id": i, "name": name} for i, name in enumerate(class_list)]
    }
    
    annotation_id = 1
    image_id = 1
    
    # Get all image files
    image_files = [f for f in os.listdir(images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"\n🔄 Converting {split_name}: {len(image_files)} images")
    
    for img_file in tqdm(image_files, desc=f"Converting {split_name}"):
        img_path = os.path.join(images_dir, img_file)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_path = os.path.join(labels_dir, label_file)
        
        if not os.path.exists(label_path):
            continue
        
        # Get image dimensions
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except Exception as e:
            tqdm.write(f"⚠️ Could not read {img_file}: {e}")
            continue
        
        # Add image
        coco_data["images"].append({
            "id": image_id,
            "file_name": img_file,
            "width": width,
            "height": height
        })
        
        # Add annotations
        try:
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # YOLO format: class_id (as integer from original dataset)
                        orig_class_id = int(parts[0])
                        
                        # Map to new class ID if needed
                        # For now, assume the class_id matches our list
                        if orig_class_id >= len(class_list):
                            tqdm.write(f"⚠️ Invalid class_id {orig_class_id} in {label_file}")
                            continue
                        
                        x_center = float(parts[1]) * width
                        y_center = float(parts[2]) * height
                        bbox_width = float(parts[3]) * width
                        bbox_height = float(parts[4]) * height
                        
                        x = x_center - bbox_width/2
                        y = y_center - bbox_height/2
                        
                        coco_data["annotations"].append({
                            "id": annotation_id,
                            "image_id": image_id,
                            "category_id": orig_class_id,
                            "bbox": [x, y, bbox_width, bbox_height],
                            "area": bbox_width * bbox_height,
                            "iscrowd": 0
                        })
                        annotation_id += 1
        except Exception as e:
            tqdm.write(f"⚠️ Error reading {label_file}: {e}")
        
        image_id += 1
    
    # Save COCO JSON
    with open(output_json, 'w') as f:
        json.dump(coco_data, f, indent=2)
    
    print(f"\n✅ {split_name} conversion complete!")
    print(f"  Images: {len(coco_data['images'])}")
    print(f"  Annotations: {len(coco_data['annotations'])}")
    
    return coco_data

# Convert each split
train_coco = yolo_to_coco(TRAIN_IMAGES, TRAIN_LABELS, f"{WORKING_DIR}/train_coco.json", class_list, class_to_id, "TRAIN")
val_coco = yolo_to_coco(VAL_IMAGES, VAL_LABELS, f"{WORKING_DIR}/val_coco.json", class_list, class_to_id, "VALIDATION")
test_coco = yolo_to_coco(TEST_IMAGES, TEST_LABELS, f"{WORKING_DIR}/test_coco.json", class_list, class_to_id, "TEST")

# Save the JSON paths for later blocks
train_json = f"{WORKING_DIR}/train_coco.json"
val_json = f"{WORKING_DIR}/val_coco.json"
test_json = f"{WORKING_DIR}/test_coco.json"

# Save paths to file
with open(f"{WORKING_DIR}/coco_paths.txt", 'w') as f:
    f.write(f"train_json={train_json}\n")
    f.write(f"val_json={val_json}\n")
    f.write(f"test_json={test_json}\n")

# Save summary
summary = {
    "train": len(train_coco['images']) if train_coco else 0,
    "validation": len(val_coco['images']) if val_coco else 0,
    "test": len(test_coco['images']) if test_coco else 0,
    "num_classes": len(class_list),
    "class_names": class_list
}

with open(f"{WORKING_DIR}/conversion_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Summary saved to {WORKING_DIR}/conversion_summary.json")
print(f"✅ COCO JSON files:")
print(f"  train_json = {train_json}")
print(f"  val_json = {val_json}")
print(f"  test_json = {test_json}")

print("\n" + "="*70)
print("✅ UPDATED BLOCK 4 COMPLETE")
print("="*70)

CONVERTING YOLO TO COCO FORMAT
📋 Classes for Faster R-CNN (3 classes):
  0: fracture
  1: no fracture
  2: pelvic_fracture

🔄 Converting TRAIN: 11840 images


Converting TRAIN: 100%|██████████| 11840/11840 [01:06<00:00, 178.68it/s]



✅ TRAIN conversion complete!
  Images: 11840
  Annotations: 23947

🔄 Converting VALIDATION: 184 images


Converting VALIDATION: 100%|██████████| 184/184 [00:01<00:00, 176.54it/s]



✅ VALIDATION conversion complete!
  Images: 184
  Annotations: 377

🔄 Converting TEST: 194 images


Converting TEST: 100%|██████████| 194/194 [00:01<00:00, 186.74it/s]


✅ TEST conversion complete!
  Images: 194
  Annotations: 412

✅ Summary saved to /kaggle/working/conversion_summary.json
✅ COCO JSON files:
  train_json = /kaggle/working/train_coco.json
  val_json = /kaggle/working/val_coco.json
  test_json = /kaggle/working/test_coco.json

✅ UPDATED BLOCK 4 COMPLETE


# **BLOCK 5: CREATE TRAIN/VAL/TEST SPLITS (80/10/10)**

In [17]:
# ============================================================================
# CLEAN BLOCK 5: REGISTER DATASET WITH UNIQUE NAME (NO CONFLICTS)
# ============================================================================

print("="*70)
print("REGISTERING DATASET WITH DETECTRON2")
print("="*70)

import time
import json
import os
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.utils.logger import setup_logger

setup_logger()

# Load paths
with open(f"{WORKING_DIR}/paths.txt", 'r') as f:
    for line in f:
        if line.startswith("TRAIN_IMAGES"):
            TRAIN_IMAGES = line.strip().split('=')[1]
        if line.startswith("VAL_IMAGES"):
            VAL_IMAGES = line.strip().split('=')[1]
        if line.startswith("TEST_IMAGES"):
            TEST_IMAGES = line.strip().split('=')[1]

# Load COCO JSON paths
with open(f"{WORKING_DIR}/coco_paths.txt", 'r') as f:
    for line in f:
        if line.startswith("train_json"):
            train_json = line.strip().split('=')[1]
        if line.startswith("val_json"):
            val_json = line.strip().split('=')[1]
        if line.startswith("test_json"):
            test_json = line.strip().split('=')[1]

print(f"📁 Train images: {TRAIN_IMAGES}")
print(f"📁 Train JSON: {train_json}")
print(f"📁 Val images: {VAL_IMAGES}")
print(f"📁 Val JSON: {val_json}")
print(f"📁 Test images: {TEST_IMAGES}")
print(f"📁 Test JSON: {test_json}")

# Load class list
with open(f"{WORKING_DIR}/class_names.json", 'r') as f:
    class_list = json.load(f)

print(f"\n📋 Classes ({len(class_list)}):")
for i, name in enumerate(class_list):
    print(f"  {i}: {name}")

# ========== CREATE A UNIQUE DATASET NAME ==========
# This avoids any conflicts with previously registered datasets
unique_id = str(int(time.time()))[-8:]
dataset_name = f"hips_pelvis_{unique_id}"

print(f"\n📝 Using unique dataset name: {dataset_name}")

# ========== REGISTER DATASETS WITH UNIQUE NAMES ==========
print("\n📝 Registering datasets...")

# Register train
if os.path.exists(train_json) and os.path.exists(TRAIN_IMAGES):
    register_coco_instances(f"{dataset_name}_train", {}, train_json, TRAIN_IMAGES)
    print(f"✅ Registered {dataset_name}_train")
else:
    print(f"❌ Cannot register train: {train_json} or {TRAIN_IMAGES} not found")

# Register validation
if os.path.exists(val_json) and os.path.exists(VAL_IMAGES):
    register_coco_instances(f"{dataset_name}_val", {}, val_json, VAL_IMAGES)
    print(f"✅ Registered {dataset_name}_val")
else:
    print(f"❌ Cannot register val: {val_json} or {VAL_IMAGES} not found")

# Register test
if os.path.exists(test_json) and os.path.exists(TEST_IMAGES):
    register_coco_instances(f"{dataset_name}_test", {}, test_json, TEST_IMAGES)
    print(f"✅ Registered {dataset_name}_test")
else:
    print(f"❌ Cannot register test: {test_json} or {TEST_IMAGES} not found")

# ========== SET METADATA ==========
print("\n📋 Setting metadata...")

# Get metadata objects and set thing_classes
MetadataCatalog.get(f"{dataset_name}_train").thing_classes = class_list
MetadataCatalog.get(f"{dataset_name}_val").thing_classes = class_list
MetadataCatalog.get(f"{dataset_name}_test").thing_classes = class_list

print("✅ Metadata set successfully!")

# ========== VERIFY REGISTRATION ==========
print("\n📊 Registered datasets:")
for split in ['train', 'val', 'test']:
    name = f"{dataset_name}_{split}"
    if name in DatasetCatalog.list():
        dataset = DatasetCatalog.get(name)
        metadata = MetadataCatalog.get(name)
        print(f"  ✅ {name}: {len(dataset)} images, {len(metadata.thing_classes)} classes")
    else:
        print(f"  ❌ {name} not found!")

# ========== SAVE DATASET NAME FOR LATER BLOCKS ==========
with open(f"{WORKING_DIR}/dataset_name.txt", 'w') as f:
    f.write(dataset_name)

print(f"\n✅ Dataset name saved to {WORKING_DIR}/dataset_name.txt")
print(f"   Name: {dataset_name}")

print("\n" + "="*70)
print("✅ CLEAN BLOCK 5 COMPLETE")
print("="*70)

REGISTERING DATASET WITH DETECTRON2
📁 Train images: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis/train/images
📁 Train JSON: /kaggle/working/train_coco.json
📁 Val images: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis/valid/images
📁 Val JSON: /kaggle/working/val_coco.json
📁 Test images: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis/test/images
📁 Test JSON: /kaggle/working/test_coco.json

📋 Classes (3):
  0: fracture
  1: no fracture
  2: pelvic_fracture

📝 Using unique dataset name: hips_pelvis_74309373

📝 Registering datasets...
✅ Registered hips_pelvis_74309373_train
✅ Registered hips_pelvis_74309373_val
✅ Registered hips_pelvis_74309373_test

📋 Setting metadata...
✅ Metadata set successfully!

📊 Registered datasets:
WARNING [03/23 23:42:53 d2.data.datasets.coco]: 
Category ids in annotations are not i

# **BLOCK 6: REGISTER DATASET WITH DETECTRON2**

In [20]:
# ============================================================================
# FINAL FIXED BLOCK 6: REGISTER DATASET WITH UNIQUE NAMES
# ============================================================================

print("="*70)
print("REGISTERING DATASET WITH DETECTRON2")
print("="*70)

import json
import os
import time
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.utils.logger import setup_logger

setup_logger()

# ========== LOAD ALL REQUIRED PATHS ==========
# Load working directory
WORKING_DIR = "/kaggle/working"

# Load paths from previous blocks
with open(f"{WORKING_DIR}/paths.txt", 'r') as f:
    for line in f:
        if line.startswith("TRAIN_IMAGES"):
            TRAIN_IMAGES = line.strip().split('=')[1]
        if line.startswith("VAL_IMAGES"):
            VAL_IMAGES = line.strip().split('=')[1]
        if line.startswith("TEST_IMAGES"):
            TEST_IMAGES = line.strip().split('=')[1]

# Load COCO JSON paths
with open(f"{WORKING_DIR}/coco_paths.txt", 'r') as f:
    for line in f:
        if line.startswith("train_json"):
            train_json = line.strip().split('=')[1]
        if line.startswith("val_json"):
            val_json = line.strip().split('=')[1]
        if line.startswith("test_json"):
            test_json = line.strip().split('=')[1]

print(f"📁 Train images: {TRAIN_IMAGES}")
print(f"📁 Train JSON: {train_json}")
print(f"📁 Val images: {VAL_IMAGES}")
print(f"📁 Val JSON: {val_json}")
print(f"📁 Test images: {TEST_IMAGES}")
print(f"📁 Test JSON: {test_json}")

# Load class list
with open(f"{WORKING_DIR}/class_names.json", 'r') as f:
    class_names = json.load(f)

if isinstance(class_names, dict):
    class_list = [class_names[i] for i in sorted(class_names.keys())]
else:
    class_list = class_names

print(f"\n📋 Classes ({len(class_list)}):")
for i, name in enumerate(class_list):
    print(f"  {i}: {name}")

# ========== CREATE UNIQUE DATASET NAMES ==========
# Use timestamp to create unique names
timestamp = str(int(time.time()))[-8:]
dataset_name = f"hips_pelvis_{timestamp}"

print(f"\n📝 Using unique dataset name: {dataset_name}")

# ========== REGISTER DATASETS WITH UNIQUE NAMES ==========
print("\n📝 Registering datasets...")

# Check if files exist before registering
if os.path.exists(train_json) and os.path.exists(TRAIN_IMAGES):
    register_coco_instances(f"{dataset_name}_train", {}, train_json, TRAIN_IMAGES)
    print(f"✅ Registered {dataset_name}_train")
else:
    print(f"❌ Cannot register train: {train_json} or {TRAIN_IMAGES} not found")

if os.path.exists(val_json) and os.path.exists(VAL_IMAGES):
    register_coco_instances(f"{dataset_name}_val", {}, val_json, VAL_IMAGES)
    print(f"✅ Registered {dataset_name}_val")
else:
    print(f"❌ Cannot register val: {val_json} or {VAL_IMAGES} not found")

if os.path.exists(test_json) and os.path.exists(TEST_IMAGES):
    register_coco_instances(f"{dataset_name}_test", {}, test_json, TEST_IMAGES)
    print(f"✅ Registered {dataset_name}_test")
else:
    print(f"❌ Cannot register test: {test_json} or {TEST_IMAGES} not found")

# ========== SET METADATA ==========
print("\n📋 Setting metadata...")
MetadataCatalog.get(f"{dataset_name}_train").thing_classes = class_list
MetadataCatalog.get(f"{dataset_name}_val").thing_classes = class_list
MetadataCatalog.get(f"{dataset_name}_test").thing_classes = class_list

print("✅ Metadata set successfully!")

# ========== VERIFY REGISTRATION ==========
print("\n📊 Registered datasets:")
for split in ['train', 'val', 'test']:
    name = f"{dataset_name}_{split}"
    if name in DatasetCatalog.list():
        dataset = DatasetCatalog.get(name)
        metadata = MetadataCatalog.get(name)
        print(f"  ✅ {name}: {len(dataset)} images, {len(metadata.thing_classes)} classes")
        print(f"     Image root: {metadata.image_root}")
    else:
        print(f"  ❌ {name} not found!")

# ========== SAVE DATASET NAME FOR LATER BLOCKS ==========
with open(f"{WORKING_DIR}/dataset_name.txt", 'w') as f:
    f.write(dataset_name)

print(f"\n✅ Dataset name saved to {WORKING_DIR}/dataset_name.txt")
print(f"   Name: {dataset_name}")

print("\n" + "="*70)
print("✅ FINAL FIXED BLOCK 6 COMPLETE")
print("="*70)

REGISTERING DATASET WITH DETECTRON2
📁 Train images: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis/train/images
📁 Train JSON: /kaggle/working/train_coco.json
📁 Val images: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis/valid/images
📁 Val JSON: /kaggle/working/val_coco.json
📁 Test images: /kaggle/input/datasets/mohamedahmed12232/hips-and-pelvis-fraction-detection/Hips_and_Pelvis/Hips and Pelvis/test/images
📁 Test JSON: /kaggle/working/test_coco.json

📋 Classes (3):
  0: fracture
  1: no fracture
  2: pelvic_fracture

📝 Using unique dataset name: hips_pelvis_74309478

📝 Registering datasets...
✅ Registered hips_pelvis_74309478_train
✅ Registered hips_pelvis_74309478_val
✅ Registered hips_pelvis_74309478_test

📋 Setting metadata...
✅ Metadata set successfully!

📊 Registered datasets:
WARNING [03/23 23:44:38 d2.data.datasets.coco]: 
Category ids in annotations are not i

# **BLOCK 7: CONFIGURE FASTER R-CNN (OPTIMIZED FOR MEDICAL IMAGING)**

In [21]:
# ============================================================================
# UPDATED BLOCK 7: CONFIGURE FASTER R-CNN
# ============================================================================

print("="*70)
print("CONFIGURING FASTER R-CNN MODEL")
print("="*70)

from detectron2.config import get_cfg
from detectron2 import model_zoo

# Load dataset name
dataset_name_file = f"{WORKING_DIR}/dataset_name.txt"
if os.path.exists(dataset_name_file):
    with open(dataset_name_file, 'r') as f:
        dataset_name = f.read().strip()
    print(f"📊 Using dataset name: {dataset_name}")
else:
    print("⚠️ dataset_name.txt not found! Using default.")
    dataset_name = "hips_pelvis"

# Load class list
with open(f"{WORKING_DIR}/class_names.json", 'r') as f:
    class_list = json.load(f)

num_classes = len(class_list)
print(f"📊 Number of classes: {num_classes}")
print(f"📋 Classes: {class_list}")

# Create configuration
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))

# Dataset - using our unique registered names
cfg.DATASETS.TRAIN = (f"{dataset_name}_train",)
cfg.DATASETS.TEST = (f"{dataset_name}_val",)

# Classes
cfg.MODEL.ROI_HEADS.NUM_CLASSES = num_classes

# ========== FASTER R-CNN OPTIMIZATIONS ==========
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.000125
cfg.SOLVER.MAX_ITER = 15000
cfg.SOLVER.STEPS = (10000, 13000)
cfg.SOLVER.WEIGHT_DECAY = 0.0005
cfg.SOLVER.CHECKPOINT_PERIOD = 2500

# Detection thresholds
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.5

# Multi-scale anchors
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32, 64, 128, 256, 512]]

# Output directory
cfg.OUTPUT_DIR = f"{WORKING_DIR}/faster_rcnn_model"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print(f"\n⚙️ FASTER R-CNN Configuration:")
print(f"  🏗️ Model: Faster R-CNN with ResNet-50 FPN")
print(f"  📋 Classes: {cfg.MODEL.ROI_HEADS.NUM_CLASSES}")
print(f"  📦 Batch size: {cfg.SOLVER.IMS_PER_BATCH}")
print(f"  🎯 Max iterations: {cfg.SOLVER.MAX_ITER}")
print(f"  📉 Learning rate: {cfg.SOLVER.BASE_LR}")
print(f"  💾 Checkpoint period: {cfg.SOLVER.CHECKPOINT_PERIOD}")
print(f"  📁 Output: {cfg.OUTPUT_DIR}")

# Save config
with open(f"{cfg.OUTPUT_DIR}/config.yaml", 'w') as f:
    f.write(cfg.dump())

print("\n✅ Configuration saved!")

print("\n" + "="*70)
print("✅ UPDATED BLOCK 7 COMPLETE")
print("="*70)

CONFIGURING FASTER R-CNN MODEL
📊 Using dataset name: hips_pelvis_74309478
📊 Number of classes: 3
📋 Classes: ['fracture', 'no fracture', 'pelvic_fracture']

⚙️ FASTER R-CNN Configuration:
  🏗️ Model: Faster R-CNN with ResNet-50 FPN
  📋 Classes: 3
  📦 Batch size: 4
  🎯 Max iterations: 15000
  📉 Learning rate: 0.000125
  💾 Checkpoint period: 2500
  📁 Output: /kaggle/working/faster_rcnn_model

✅ Configuration saved!

✅ UPDATED BLOCK 7 COMPLETE


# **BLOCK 8: TRAIN FASTER R-CNN MODEL**

In [22]:
# ============================================================================
# UPDATED BLOCK 8: TRAIN FASTER R-CNN MODEL
# ============================================================================

print("="*70)
print("TRAINING FASTER R-CNN MODEL")
print("⚠️ This will take several hours!")
print("="*70)

import time
import glob
import re
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
from detectron2.checkpoint import DetectionCheckpointer

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, True, output_folder)

# Load dataset name
with open(f"{WORKING_DIR}/dataset_name.txt", 'r') as f:
    dataset_name = f.read().strip()

print(f"📊 Training on dataset: {dataset_name}")

# Check for existing checkpoints
checkpoint_files = glob.glob(os.path.join(cfg.OUTPUT_DIR, "model_*.pth"))
checkpoint_files.sort(key=os.path.getmtime, reverse=True)

resume_from = False
latest_iter = 0

if checkpoint_files:
    latest_checkpoint = checkpoint_files[0]
    print(f"\n✅ Found checkpoint: {os.path.basename(latest_checkpoint)}")
    
    match = re.search(r'model_(\d+)\.pth', latest_checkpoint)
    if match:
        latest_iter = int(match.group(1))
        print(f"📊 This checkpoint is from iteration: {latest_iter}")
        
        if latest_iter > 100:
            resume_from = True
            print(f"🔄 Will RESUME training from iteration {latest_iter}")
        else:
            print(f"⚠️ Checkpoint too early, starting fresh")
    else:
        resume_from = True
else:
    print("\n🔄 No checkpoints found. Starting training from scratch.")

# Initialize trainer
trainer = CustomTrainer(cfg)

if resume_from:
    print(f"\n📥 Loading checkpoint: {latest_checkpoint}")
    DetectionCheckpointer(trainer.model).load(latest_checkpoint)
    print("✅ Checkpoint loaded successfully!")
else:
    print("\n🆕 Starting fresh training...")
    trainer.resume_or_load(resume=False)

# Check GPU
print(f"\n🎮 GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Start training
start_time = time.time()
print(f"\n🔥 Training started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Starting from iteration: ~{latest_iter} of {cfg.SOLVER.MAX_ITER}")

try:
    trainer.train()
    
    end_time = time.time()
    duration = (end_time - start_time) / 3600
    
    print("\n" + "="*70)
    print("✅ FASTER R-CNN TRAINING COMPLETE!")
    print("="*70)
    print(f"⏱️  Training time: {duration:.2f} hours")
    print(f"💾 Final model: {cfg.OUTPUT_DIR}/model_final.pth")
    
except KeyboardInterrupt:
    print("\n\n⚠️ Training interrupted by user!")
    print(f"✅ Progress saved in: {cfg.OUTPUT_DIR}")
    
except Exception as e:
    print(f"\n❌ Training error: {e}")
    print(f"✅ Checkpoints saved in: {cfg.OUTPUT_DIR}")

print("\n" + "="*70)
print("✅ UPDATED BLOCK 8 COMPLETE")
print("="*70)

TRAINING FASTER R-CNN MODEL
⚠️ This will take several hours!
📊 Training on dataset: hips_pelvis_74309478

🔄 No checkpoints found. Starting training from scratch.
[03/23 23:45:07 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
         

R-50.pkl: 102MB [00:00, 131MB/s]                             


[03/23 23:45:08 d2.checkpoint.c2_model_loading]: Renaming Caffe2 weights ......
[03/23 23:45:08 d2.checkpoint.c2_model_loading]: Following weights matched with submodule backbone.bottom_up - Total num: 54


Some model parameters or buffers are not found in the checkpoint:
backbone.fpn_lateral2.{bias, weight}
backbone.fpn_lateral3.{bias, weight}
backbone.fpn_lateral4.{bias, weight}
backbone.fpn_lateral5.{bias, weight}
backbone.fpn_output2.{bias, weight}
backbone.fpn_output3.{bias, weight}
backbone.fpn_output4.{bias, weight}
backbone.fpn_output5.{bias, weight}
proposal_generator.rpn_head.anchor_deltas.{bias, weight}
proposal_generator.rpn_head.conv.{bias, weight}
proposal_generator.rpn_head.objectness_logits.{bias, weight}
roi_heads.box_head.fc1.{bias, weight}
roi_heads.box_head.fc2.{bias, weight}
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
The checkpoint state_dict contains keys that are not used by the model:
  fc1000.{bias, weight}
  stem.conv1.bias



🎮 GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4

🔥 Training started at: 2026-03-23 23:45:08
📊 Starting from iteration: ~0 of 15000
[03/23 23:45:08 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0323 23:45:13.135000 55 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[03/23 23:45:27 d2.utils.events]:  eta: 2:59:33  iter: 19  total_loss: 2.441  loss_cls: 1.364  loss_box_reg: 0.08254  loss_rpn_cls: 0.6781  loss_rpn_loc: 0.3273    time: 0.6835  last_time: 0.7321  data_time: 0.0233  last_data_time: 0.0092   lr: 2.4976e-06  max_mem: 3065M


2026-03-23 23:45:32.387786: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774309532.788042      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774309532.890906      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774309533.880084      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774309533.880119      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774309533.880122      55 computation_placer.cc:177] computation placer alr

[03/23 23:46:16 d2.utils.events]:  eta: 3:01:16  iter: 39  total_loss: 2.094  loss_cls: 1.045  loss_box_reg: 0.08347  loss_rpn_cls: 0.678  loss_rpn_loc: 0.2934    time: 0.7029  last_time: 0.6430  data_time: 0.0130  last_data_time: 0.0108   lr: 4.9951e-06  max_mem: 3065M
[03/23 23:46:31 d2.utils.events]:  eta: 3:01:01  iter: 59  total_loss: 1.745  loss_cls: 0.6468  loss_box_reg: 0.1003  loss_rpn_cls: 0.6755  loss_rpn_loc: 0.3294    time: 0.7046  last_time: 0.7540  data_time: 0.0109  last_data_time: 0.0106   lr: 7.4926e-06  max_mem: 3065M
[03/23 23:46:45 d2.utils.events]:  eta: 3:01:33  iter: 79  total_loss: 1.462  loss_cls: 0.4007  loss_box_reg: 0.104  loss_rpn_cls: 0.6666  loss_rpn_loc: 0.2663    time: 0.7112  last_time: 0.6207  data_time: 0.0104  last_data_time: 0.0106   lr: 9.9901e-06  max_mem: 3699M
[03/23 23:47:00 d2.utils.events]:  eta: 3:02:17  iter: 99  total_loss: 1.353  loss_cls: 0.3068  loss_box_reg: 0.121  loss_rpn_cls: 0.6666  loss_rpn_loc: 0.2583    time: 0.7164  last_time

# **BLOCK 8: CONFIGURE FASTER R-CNN**

# **BLOCK 9: TRAIN MODEL**

# **BLOCK 10: EVALUATE & SAVE**

In [ ]:
# ============================================================================
# BLOCK 10: EVALUATE AND SAVE
# ============================================================================

from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# Load best model
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)

# Evaluate
evaluator = COCOEvaluator("fracatlas_test", cfg, False, output_dir=cfg.OUTPUT_DIR)
test_loader = build_detection_test_loader(cfg, "fracatlas_test")
results = inference_on_dataset(predictor.model, test_loader, evaluator)

print("\n📊 TEST RESULTS:")
for metric, value in results['bbox'].items():
    print(f"  {metric}: {value:.2f}")

# Save results
with open(f"{cfg.OUTPUT_DIR}/test_results.json", 'w') as f:
    json.dump(results, f, indent=2)

# Package final model
!mkdir -p /kaggle/working/deployment
!cp {cfg.OUTPUT_DIR}/model_final.pth /kaggle/working/deployment/
!cp {cfg.OUTPUT_DIR}/config.yaml /kaggle/working/deployment/

print("\n✅ Final model saved to /kaggle/working/deployment/")
print("📥 Download from: Kaggle → Output → working/deployment/")